# 🤖 Phase 10: Machine Learning Churn Prediction
**Project:** AI-Powered Customer Intelligence & Revenue Optimization Platform  
**Analyst:** Naman Roy  

---
### Objective
Build a predictive classification model using Random Forest to assess churn risk based on behavioral features without target leakage.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

proc_dir = '../../02_Data/03_Processed_Data'
cust_features = pd.read_csv(os.path.join(proc_dir, 'customer_features.csv'))

# Define Churn as inactivity > 180 days
cust_features['is_churned'] = (cust_features['recency_days'] > 180).astype(int)

feature_cols = [
    'total_orders', 'total_spend', 'avg_order_value', 'total_items',
    'avg_freight', 'freight_ratio', 'avg_review_score', 'avg_delivery_delay', 'late_order_ratio'
]

# Impute minor missing values
cust_features['avg_review_score'] = cust_features['avg_review_score'].fillna(cust_features['avg_review_score'].mean())
cust_features['avg_delivery_delay'] = cust_features['avg_delivery_delay'].fillna(0)
cust_features['late_order_ratio'] = cust_features['late_order_ratio'].fillna(0)

X = cust_features[feature_cols]
y = cust_features['is_churned']

print(f"Total Customers: {len(X):,}")
print(f"Churn Rate     : {y.mean()*100:.1f}%")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.3f}")


In [ ]:
# Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 4.5))
sns.barplot(x=importances.values, y=importances.index, color='#2563EB')
plt.title('Random Forest Feature Importance — Churn Risk Factors', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(importances)
